# 02 — Modelagem e carga: do relacional para o grafo

Este é o notebook principal do material. Vamos:

1. Olhar o modelo relacional que geramos no notebook 01 e achar onde ele trava
2. Traduzir esse modelo para um **modelo de grafo**
3. Criar constraints no Neo4j
4. Carregar os dados em lotes

> Pré-requisito: você já rodou o notebook `01` (que gerou os CSVs) e já tem um
> banco Neo4j em mãos (veja o `README.md`). A primeira célula de código detecta
> onde os CSVs estão — no seu Drive, se você estiver no Colab.

## 1. O modelo relacional e seus dois nós cegos

| Tabela | Chave primária | Colunas |
|---|---|---|
| `clientes` | `cpf` | `nome`, `data_cadastro`, `rg`, `email`, `telefone` |
| `bancos` | `codigo` | `nome` |
| `empresas` | `cnpj` | `nome` |
| `transacoes` | `transacao_id` | `tipo`, `valor`, `step`, `ts`, `origem_cpf`, `destino_id` |

Quatro tabelas bem normais. E dois problemas que só aparecem quando você começa a
fazer perguntas interessantes.

### Nó cego #1: identidade é coluna, e coluna não se conecta

O `rg` do cliente é uma coluna — um pedaço de texto solto dentro da linha. Se dois
clientes tiverem o mesmo RG, o banco de dados **não tem opinião nenhuma sobre
isso**: são só duas linhas que por coincidência guardam a mesma string.

E, diferente do CPF, não há nada impedindo: o RG não tem unicidade nacional e
raramente é validado no cadastro. Mesma história para e-mail e telefone.

Para descobrir esse reaproveitamento em SQL você precisa perguntar campo por campo:

```sql
SELECT rg,       COUNT(*) FROM clientes GROUP BY rg       HAVING COUNT(*) > 1;
SELECT email,    COUNT(*) FROM clientes GROUP BY email    HAVING COUNT(*) > 1;
SELECT telefone, COUNT(*) FROM clientes GROUP BY telefone HAVING COUNT(*) > 1;
```

Três consultas para três campos. Adicione "endereço" ao cadastro e será uma quarta.
E, pior: cada consulta só acha o que você já desconfiava de procurar.

### Nó cego #2: a chave estrangeira polimórfica

`transacoes.destino_id` aponta para `clientes`, `bancos` **ou** `empresas`,
dependendo do valor de `tipo`. O banco não pode garantir integridade referencial
nisso — não existe uma FK que aponte para "uma de três tabelas". Você fica entre
criar três colunas de FK (duas sempre `NULL`) ou abrir mão da garantia.

Guarde os dois. É exatamente isso que o modelo de grafo desfaz.

## 2. O modelo de grafo

| Do relacional... | ...para o grafo |
|---|---|
| linha de `clientes` | nó `(:Cliente {cpf, nome, data_cadastro})` — CPF fica como **propriedade** |
| colunas de gabarito | propriedades `gabarito_anel` / `gabarito_laranja` no `Cliente` (só para medir o acerto no notebook 04) |
| **coluna** `clientes.rg` | **nó** `(:RG {valor})` + relacionamento `(:Cliente)-[:TEM_RG]->(:RG)` |
| **coluna** `clientes.email` | **nó** `(:Email {valor})` + relacionamento `[:TEM_EMAIL]` |
| **coluna** `clientes.telefone` | **nó** `(:Telefone {valor})` + relacionamento `[:TEM_TELEFONE]` |
| linha de `bancos` | nó `(:Banco {codigo, nome})` |
| linha de `empresas` | nó `(:Empresa {cnpj, nome})` — CNPJ fica como **propriedade** |
| linha de `transacoes` | nó com **duas labels**: `:Transacao` + o tipo (`:Pix`, `:Boleto`, `:Compra`, `:Deposito`, `:Saque`), com `(origem:Cliente)-[:REALIZOU]->(t)` e `(t)-[:PARA]->(destino)` |

Três decisões de modelagem, e vale entender o motivo de cada uma:

**1. Colunas de identidade "frouxa" viram nós.** Essa é a mais importante do
material. Um RG deixa de ser texto dentro de dez mil linhas e passa a ser **uma
entidade única no grafo**. Se dois clientes têm o mesmo RG, os dois apontam para o
*mesmo nó* — e o reaproveitamento deixa de ser uma coincidência de strings para se
tornar **estrutura**. Você não precisa mais perguntar "esse RG se repete?", porque
a resposta é visível: o nó tem duas setas chegando.

E note que a pergunta "quem compartilha *qualquer* identificador?" passa a ser uma
consulta só, não uma por campo — porque no grafo os três casos têm a mesma forma.

**2. O destino da transação não precisa saber seu tipo.** `(t)-[:PARA]->(destino)`
funciona igual se o destino for `Cliente`, `Banco` ou `Empresa`: o próprio nó
carrega seu label. A FK polimórfica simplesmente deixa de ser um problema.

**3. O tipo da transação vira label, não propriedade.** Cada transação carrega duas
labels ao mesmo tempo (`:Transacao:Pix`) — a genérica para perguntar sobre todas, a
específica para filtrar um tipo sem `WHERE`. É o mesmo desenho do [dataset original
do guia Neo4j](https://github.com/neo4j-graph-examples/fraud-detection), que modela
`CashIn`/`CashOut`/`Payment`/`Debit`/`Transfer` assim.

### Espera: por que o CPF virou nó e o CNPJ não?

Essa é a pergunta certa a fazer, e a resposta é a lição mais importante de
modelagem de grafos que este material tem a oferecer.

CPF e CNPJ são a mesma *categoria* de coisa: identificadores nacionais, únicos,
atribuídos pelo governo. Se a modelagem seguisse o tipo do dado, os dois deveriam
virar nós. Mas eles foram modelados de formas opostas:

| | CPF | CNPJ |
|---|---|---|
| No grafo | **nó** `(:RG)` | **propriedade** de `(:Empresa)` |
| Compartilhar é... | um sinal de fraude | um erro de cadastro |
| A pergunta que interessa | "quem mais usa esse CPF?" | "qual é o CNPJ desta empresa?" |

O CPF virou nó porque **queremos perguntar sobre o compartilhamento dele**. Duas
pessoas com o mesmo CPF é o padrão que estamos caçando; transformá-lo em nó torna
esse padrão visível na estrutura.

O CNPJ não. Uma empresa tem um CNPJ, e duas empresas com o mesmo CNPJ seriam a
mesma empresa cadastrada duas vezes — um bug, não um caso de investigação. Nunca
vamos perguntar "quem mais usa este CNPJ?", então transformá-lo em nó só
adicionaria 30 nós e 30 relacionamentos sem responder pergunta nenhuma.

**A regra prática:** transforme um atributo em nó quando quiser perguntar *quem
mais se conecta a ele*. Se você só quer ler o valor a partir do dono, deixe como
propriedade.

Modelagem de grafos não segue o tipo do dado. Segue as **perguntas** que você
pretende fazer.

In [ ]:
!pip install -q neo4j-rust-ext pandas python-dotenv

## Onde ficam os CSVs

Este notebook detecta sozinho onde está rodando:

- **No Google Colab** — monta o seu Google Drive e usa a pasta
  `workshop-neo4j-csv` dentro dele. Assim os arquivos **sobrevivem** ao fim da
  sessão: o Colab apaga o disco local quando o runtime é reciclado, mas o que está
  no Drive fica. É o que permite gerar os dados hoje e recarregar amanhã.
- **Localmente** — usa a pasta `data/` ao lado do notebook.

No Colab, a célula abaixo vai abrir um pedido de autorização do Google. É o próprio
Colab pedindo acesso ao seu Drive; sem isso ele não consegue gravar lá.

In [ ]:
import os

try:
    from google.colab import drive
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    drive.mount("/content/drive")
    # o Drive do usuário fica em MyDrive — escrever na raiz da montagem
    # (/content/drive) não sincroniza com o Drive de verdade
    raiz_drive = "/content/drive/MyDrive"
    if not os.path.isdir(raiz_drive):
        raiz_drive = "/content/drive/My Drive"   # nome antigo, com espaço
    PASTA_DADOS = os.path.join(raiz_drive, "workshop-neo4j-csv")
else:
    PASTA_DADOS = "data"

os.makedirs(PASTA_DADOS, exist_ok=True)

print(f"Ambiente: {'Google Colab (dados no seu Drive)' if EM_COLAB else 'local'}")
print(f"Pasta dos CSVs: {PASTA_DADOS}")

## Credenciais

Este notebook busca as credenciais em três lugares, na ordem:

1. **Secrets do Colab** — o ícone de chave 🔑 na barra lateral esquerda.
2. **Variáveis de ambiente** — incluindo um arquivo `.env` na pasta do projeto
   (copie o `.env.example` e preencha). É o caminho para quem roda localmente.
3. **Pergunta na tela** — se não achou nas opções acima, pergunta aqui mesmo.

> ⚠️ **No Colab, cada notebook precisa de permissão para cada secret.** Ter criado
> o secret na sua conta não basta: abra o painel 🔑 e ative a chave
> **"Acesso ao notebook"** (*Notebook access*) para **este** notebook. Sem isso o
> secret é ignorado silenciosamente e o notebook volta a perguntar na tela.
>
> Se os secrets estiverem configurados (e liberados), a célula abaixo não pergunta
> nada — apenas conecta.

Além de poupar digitação, as duas primeiras opções evitam que a URI da sua
instância fique gravada na saída da célula caso você comite o notebook.

In [ ]:
import os
from getpass import getpass

try:  # carrega um arquivo .env, se existir
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass  # python-dotenv não instalado, ou não há .env — segue o baile


def credencial(nome, prompt, secreta=False, padrao=None):
    """Busca em: Secrets do Colab > variável de ambiente (.env) > pergunta na tela."""
    try:
        from google.colab import userdata
        if valor := userdata.get(nome):
            return valor
    except ImportError:
        pass  # não estamos no Colab
    except Exception as e:
        # O caso confuso: o secret existe, mas este notebook não tem permissão.
        # Sem este aviso, o notebook só voltaria a perguntar, sem explicar por quê.
        if "NotebookAccess" in type(e).__name__:
            print(f"⚠️  O secret '{nome}' existe, mas este notebook não tem acesso a ele.")
            print(f"    Abra o painel 🔑 e ative 'Acesso ao notebook' para '{nome}'.")

    if valor := os.environ.get(nome):
        return valor

    return (getpass(prompt) if secreta else input(prompt)).strip() or padrao


NEO4J_URI = credencial("NEO4J_URI", "URI do Neo4j (ex.: neo4j+s://xxxx.databases.neo4j.io): ")
NEO4J_USER = credencial("NEO4J_USERNAME", "Usuário [neo4j]: ", padrao="neo4j")
NEO4J_PASSWORD = credencial("NEO4J_PASSWORD", "Senha: ", secreta=True)
# Atenção: em instâncias AuraDB recentes o banco NÃO se chama "neo4j", e sim o
# próprio instance id (o prefixo da URI). Confira em Aura Console > sua instância,
# ou rode SHOW DATABASES.
NEO4J_DATABASE = credencial("NEO4J_DATABASE", "Nome do banco (geralmente = instance id) [neo4j]: ", padrao="neo4j")

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Conectado!")

## 3. Criando constraints

Constraints de unicidade fazem duas coisas ao mesmo tempo: garantem que `MERGE`
não crie duplicatas e criam um índice de brinde — sem o índice, cada
`MERGE`/`MATCH` varre todos os nós daquele label, e a carga fica lenta de um jeito
que piora conforme os dados crescem.

A constraint de `Transacao` é declarada só na label genérica: como todo nó de
transação carrega essa label além da específica, isso já garante `transacao_id`
único em todos os tipos.

In [ ]:
constraints = [
    "CREATE CONSTRAINT cliente_cpf IF NOT EXISTS FOR (c:Cliente) REQUIRE c.cpf IS UNIQUE",
    "CREATE CONSTRAINT rg_valor IF NOT EXISTS FOR (r:RG) REQUIRE r.valor IS UNIQUE",
    "CREATE CONSTRAINT email_valor IF NOT EXISTS FOR (e:Email) REQUIRE e.valor IS UNIQUE",
    "CREATE CONSTRAINT telefone_valor IF NOT EXISTS FOR (t:Telefone) REQUIRE t.valor IS UNIQUE",
    "CREATE CONSTRAINT banco_codigo IF NOT EXISTS FOR (b:Banco) REQUIRE b.codigo IS UNIQUE",
    "CREATE CONSTRAINT empresa_cnpj IF NOT EXISTS FOR (e:Empresa) REQUIRE e.cnpj IS UNIQUE",
    "CREATE CONSTRAINT transacao_id IF NOT EXISTS FOR (t:Transacao) REQUIRE t.transacao_id IS UNIQUE",
]

for stmt in constraints:
    driver.execute_query(stmt, database_=NEO4J_DATABASE)

print(f"{len(constraints)} constraints criadas (ou já existentes).")

## 4. Função auxiliar de carga em lote

Nunca envie uma linha por vez para o banco — isso significaria uma viagem de rede
por linha. O padrão certo é `UNWIND $rows AS row` dentro da query: você manda um
lote (aqui, 2.000 linhas) e o Cypher itera sobre ele do lado do servidor. Lotes
maiores significam menos idas e voltas de rede — e é **isso** que domina o tempo
desta carga.

Vale saber onde o tempo realmente vai. Medindo a carga das 140 mil transações
numa instância Free:

| | Tempo |
|---|---|
| Serialização dos dados no cliente | ~1,7 s (ou ~0,1 s com o `neo4j-rust-ext`) |
| Rede + escrita no servidor | ~60 s |

Ou seja: a extensão em Rust acelera a serialização em mais de 10×, mas isso muda
só uns 2% do total, porque o gargalo é a viagem até o servidor. Aumentar o tamanho
do lote ataca o gargalo certo; trocar o driver, não. É um bom lembrete de medir
antes de otimizar.

Usamos `driver.execute_query(...)` — a forma mais simples de mandar uma query pro
Neo4j pelo driver Python: você passa a query, os parâmetros (aqui, `rows=lote`) e
qual banco (`database_`), e ele cuida de abrir sessão, transação e commit.

In [ ]:
def carregar_em_lotes(query, registros, batch_size=2_000, rotulo=""):
    total = len(registros)
    for inicio in range(0, total, batch_size):
        lote = registros[inicio: inicio + batch_size]
        driver.execute_query(query, rows=lote, database_=NEO4J_DATABASE)
    print(f"{rotulo}: {total} registros carregados")

## 5. Carregando os nós simples: Cliente, Banco, Empresa

Uma linha de tabela vira um nó. É a parte mais direta da tradução — note que
carregamos só as colunas de "cadastro" (`nome`, `data_cadastro`); CPF, e-mail e
telefone ficam para o passo seguinte, porque eles não vão virar propriedades.

In [ ]:
import pandas as pd

# dtype=str em tudo: essas tabelas são só identificadores e texto
df_clientes = pd.read_csv(f"{PASTA_DADOS}/clientes.csv", dtype=str).fillna("")
df_bancos = pd.read_csv(f"{PASTA_DADOS}/bancos.csv", dtype=str)
df_empresas = pd.read_csv(f"{PASTA_DADOS}/empresas.csv", dtype=str)

print(df_bancos.to_dict("records"))  # confira: os códigos mantiveram os zeros à esquerda

carregar_em_lotes(
    """
    UNWIND $rows AS row
    MERGE (c:Cliente {cpf: row.cpf})
    SET c.nome = row.nome, c.data_cadastro = row.data_cadastro
    // colunas de conferência: só existem porque os dados são sintéticos.
    // Nenhuma consulta de detecção usa isso — só a medição final do notebook 04.
    SET c.gabarito_anel = CASE row.gabarito_anel WHEN '' THEN null
                          ELSE toInteger(row.gabarito_anel) END,
        c.gabarito_laranja = (row.gabarito_laranja = 'True')
    """,
    df_clientes.to_dict("records"),
    rotulo="Cliente",
)

carregar_em_lotes(
    """
    UNWIND $rows AS row
    MERGE (b:Banco {codigo: row.codigo})
    SET b.nome = row.nome
    """,
    df_bancos.to_dict("records"),
    rotulo="Banco",
)

carregar_em_lotes(
    """
    UNWIND $rows AS row
    MERGE (e:Empresa {cnpj: row.cnpj})
    SET e.nome = row.nome
    """,
    df_empresas.to_dict("records"),
    rotulo="Empresa",
)

## 6. Transformando colunas em nós

Aqui é onde a mágica acontece, e ela cabe em duas palavras: `MERGE` em vez de
`CREATE`.

`MERGE (i:RG {valor: row.rg})` significa *"ache o nó com esse RG; se não existir,
crie"*. Então, quando o segundo cliente de um anel de fraude chegar com o mesmo RG,
o `MERGE` **encontra** o nó que o primeiro criou em vez de duplicá-lo — e passa a
apontar para ele.

O reaproveitamento de identidade, que na tabela era uma coincidência invisível de
strings, vira um nó com várias setas chegando. Sem nenhuma detecção, sem nenhuma
regra: só como consequência de ter modelado identidade como entidade.

In [ ]:
registros_clientes = df_clientes.to_dict("records")

carregar_em_lotes(
    """
    UNWIND $rows AS row
    MATCH (c:Cliente {cpf: row.cpf})
    MERGE (i:RG {valor: row.rg})
    MERGE (c)-[:TEM_RG]->(i)
    """,
    registros_clientes,
    rotulo="TEM_RG",
)

carregar_em_lotes(
    """
    UNWIND $rows AS row
    MATCH (c:Cliente {cpf: row.cpf})
    MERGE (i:Email {valor: row.email})
    MERGE (c)-[:TEM_EMAIL]->(i)
    """,
    registros_clientes,
    rotulo="TEM_EMAIL",
)

carregar_em_lotes(
    """
    UNWIND $rows AS row
    MATCH (c:Cliente {cpf: row.cpf})
    MERGE (i:Telefone {valor: row.telefone})
    MERGE (c)-[:TEM_TELEFONE]->(i)
    """,
    registros_clientes,
    rotulo="TEM_TELEFONE",
)

### A primeira evidência

Carregamos 10.000 clientes, cada um com um RG, um e-mail e um telefone. Se todos
fossem únicos, teríamos exatamente 10.000 nós de cada tipo. Vamos contar:

In [ ]:
records, _, _ = driver.execute_query("""
    MATCH (c:Cliente) WITH count(c) AS clientes
    MATCH (i:RG) WITH clientes, count(i) AS rgs
    MATCH (i:Email) WITH clientes, rgs, count(i) AS emails
    MATCH (i:Telefone) RETURN clientes, rgs, emails, count(i) AS telefones
""", database_=NEO4J_DATABASE)

r = records[0]
print(f"Clientes:  {r['clientes']}")
print(f"RGs:       {r['rgs']:>5}  ({r['clientes'] - r['rgs']} a menos que o esperado)")
print(f"E-mails:   {r['emails']:>5}  ({r['clientes'] - r['emails']} a menos)")
print(f"Telefones: {r['telefones']:>5}  ({r['clientes'] - r['telefones']} a menos)")

Faltam nós — e cada nó "faltando" é um identificador que **duas ou mais pessoas
estão usando**.

Essa diferença é a fraude aparecendo sozinha, como efeito colateral da carga. Não
escrevemos nenhuma consulta de detecção, nenhum `GROUP BY ... HAVING`. Só
carregamos os dados no formato certo.

É esse fio que o notebook 04 vai puxar.

## 7. Carregando as transações

Como o `tipo` decide tanto a label da transação quanto em qual label o destino
está, fazemos a carga em cinco passadas — uma por tipo, cada uma com seu `MATCH`
de destino tipado e sua label de saída.

Esse é o último lugar em que ainda pagamos o preço do modelo relacional: é aqui que
a chave estrangeira polimórfica dá trabalho. Depois de carregado, o grafo nunca mais
precisa saber disso — `(t)-[:PARA]->(destino)` funciona igual para qualquer
destino, e `MATCH (t:Pix)` dispensa filtrar por propriedade.

In [ ]:
# Identificadores como texto; valor/step/ts continuam numéricos, porque
# nesses casos o número é o dado em si, não um rótulo.
df_transacoes = pd.read_csv(f"{PASTA_DADOS}/transacoes.csv", dtype={
    "transacao_id": str,
    "tipo": str,
    "origem_cpf": str,
    "destino_id": str,
})

query_transacao = """
UNWIND $rows AS row
MATCH (origem:Cliente {{cpf: row.origem_cpf}})
MATCH (destino:{label_destino} {{{chave_destino}: row.destino_id}})
CREATE (t:Transacao:{tipo} {{
    transacao_id: row.transacao_id, valor: row.valor,
    step: row.step, ts: row.ts, fraude_real: row.fraude_real
}})
CREATE (origem)-[:REALIZOU]->(t)
CREATE (t)-[:PARA]->(destino)
"""

TIPOS_TRANSACAO = {
    "Pix": ("Cliente", "cpf"),
    "Compra": ("Empresa", "cnpj"),
    "Deposito": ("Empresa", "cnpj"),
    "Boleto": ("Banco", "codigo"),
    "Saque": ("Empresa", "cnpj"),
}

for tipo, (label_destino, chave_destino) in TIPOS_TRANSACAO.items():
    subset = df_transacoes[df_transacoes["tipo"] == tipo]
    carregar_em_lotes(
        query_transacao.format(tipo=tipo, label_destino=label_destino, chave_destino=chave_destino),
        subset.to_dict("records"),
        rotulo=f"Transacao:{tipo}",
    )

## 8. Conferindo a carga

Esta é a etapa que as pessoas pulam — e é a que pega o erro do `dtype`. Como
linhas sem correspondência no `MATCH` desaparecem sem reclamar, a única forma de
saber se a carga foi completa é **comparar as contagens com o CSV de origem**.

Repare que `Pix`, `Boleto`, `Compra`, `Deposito` e `Saque` aparecem como labels
próprias ao lado de `Transacao` — cada transação é contada nas duas.

In [ ]:
esperado = {
    "Cliente": len(df_clientes),
    "Banco": len(df_bancos),
    "Empresa": len(df_empresas),
    "Transacao": len(df_transacoes),
}

registros, _, _ = driver.execute_query("""
    CALL db.labels() YIELD label
    CALL apoc.cypher.run('MATCH (n:`' + label + '`) RETURN count(n) AS c', {})
    YIELD value
    RETURN label, value.c AS total
""", database_=NEO4J_DATABASE)
no_grafo = {r["label"]: r["total"] for r in registros}

tudo_ok = True
for label, qtd_esperada in esperado.items():
    carregado = no_grafo.get(label, 0)
    ok = carregado == qtd_esperada
    tudo_ok = tudo_ok and ok
    print(f"{'✅' if ok else '❌'} {label:12s} CSV: {qtd_esperada:>6,}   grafo: {carregado:>6,}")

print()
if tudo_ok:
    print("Carga completa: todas as tabelas bateram com o grafo.")
else:
    print("⚠️  Alguma tabela não bateu — suspeite de conversão de tipo nos identificadores.")

In [ ]:
records, _, _ = driver.execute_query("""
    CALL db.labels() YIELD label
    CALL apoc.cypher.run('MATCH (n:`' + label + '`) RETURN count(n) AS c', {})
    YIELD value
    RETURN label, value.c AS total
    ORDER BY label
""", database_=NEO4J_DATABASE)
for r in records:
    print(f"{r['label']:15s} {r['total']}")

print()

records, _, _ = driver.execute_query("""
    CALL db.relationshipTypes() YIELD relationshipType
    CALL apoc.cypher.run('MATCH ()-[r:`' + relationshipType + '`]->() RETURN count(r) AS c', {})
    YIELD value
    RETURN relationshipType, value.c AS total
    ORDER BY relationshipType
""", database_=NEO4J_DATABASE)
for r in records:
    print(f"{r['relationshipType']:15s} {r['total']}")

## Recapitulando

Você acabou de:

1. Traduzir 4 tabelas relacionais para um grafo, transformando **colunas de
   identidade frouxa em nós** (e deixando CPF/CNPJ como propriedade) — a decisão
   que faz o resto do material funcionar
2. Criar constraints de unicidade (que são também os índices da carga)
3. Carregar tudo em lotes com `UNWIND` — o padrão para carga via driver, nunca
   linha a linha
4. Resolver a chave estrangeira polimórfica com uma passada por tipo, atribuindo a
   label correta em cada uma
5. Ver a fraude aparecer sozinha na contagem de nós, sem ter escrito nenhuma
   consulta de detecção

**Próximo passo:** `03_cypher_basico.ipynb` — aprender a linguagem de consulta com
esse grafo, antes de partir para a investigação.

In [ ]:
driver.close()